# Extracting Structure from Chaos: Insurance Claims with LangGraph + Snowflake Cortex

In this hands-on lab, you'll build a production-grade pipeline to extract structured data from unstructured insurance adjuster notes using **LangGraph** for orchestration and **Snowflake Cortex** for LLM inference.

## Learning Objectives

1. Load and explore the FEMA NFIP flood claims dataset in Snowflake
2. Design extraction schemas with Pydantic -- including fields that may not exist in the source text
3. Organize fields into logical groups for focused, parallel extraction
4. Build a LangGraph pipeline using the **Send API** for fan-out extraction with validate → fix → finalize
5. Run batch extraction with checkpointing and restartability
6. Measure extraction accuracy against ground truth

---
# Section 1: Exploring the FEMA NFIP Data

The [FEMA National Flood Insurance Program (NFIP)](https://www.fema.gov/about/openfema/data-sets#nfip) dataset contains over **2.7 million flood insurance claims** across the United States. Each row includes structured fields like damage amounts, property details, geographic info, and flood characteristics.

Our plan:
1. Explore the data to understand what we're working with
2. Create a curated sample for the lab exercises


### EXERCISE: Explore the Data

Before we start generating text and extracting data, you need to understand what you're working with. Write SQL queries to answer these questions:

1. What are the **top 10 states** by claim count?
2. What is the **average building damage amount** grouped by flood event?
3. What are the **distinct flood zone** values and how many claims are in each?
4. What is the **range of water depth** values (min, max, avg)?

Understanding these distributions will help you evaluate your extraction pipeline's accuracy later.

In [ ]:
USE DATABASE CLAIMS_EXTRACTION_LAB;
USE SCHEMA HOL;

In [ ]:
-- EXERCISE: Write your exploration queries here.
-- Hint: Use GROUP BY, COUNT(*), AVG(), MIN(), MAX()

-- 1. Top 10 states by claim count


-- 2. Average building damage by flood event (top 10 events)


-- 3. Flood zone distribution


-- 4. Water depth statistics


In [ ]:
import json
import time
import operator
from typing import TypedDict, Literal, Optional, Annotated
from datetime import datetime

from pydantic import BaseModel, Field, field_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
-- Create a small sample for iterative testing during the lab

CREATE OR REPLACE TABLE CLAIMS_SAMPLE AS
SELECT
    *
FROM (
    SELECT *
    FROM FIMA_CLAIMS
    WHERE building_damage_amount > 0
      AND contents_damage_amount > 0
      AND flood_event IS NOT NULL
      AND water_depth IS NOT NULL
      AND state IS NOT NULL
      AND flood_zone IS NOT NULL
    LIMIT 5
);

SELECT * FROM CLAIMS_SAMPLE;

---
# Section 2: Designing Extraction Schemas with Pydantic

In real-world extraction, you rarely have clean 1:1 alignment between your schema and the source text. Some fields will be explicitly stated, others require interpretation, and some might not be present at all. A good schema handles all three cases.

We'll use **Pydantic models** to:

1. Define target schemas with types, descriptions, and **field groups**
2. Generate JSON schemas for Cortex **structured output**
3. Validate extracted data and catch errors automatically

We're expanding beyond the obvious fields to ~21 total, organized into three groups:
- **Claim & Location**: identifiers and geographic info
- **Flood Event & Damage**: what happened and what it cost
- **Property Details**: building characteristics and conditions

Some fields (like `mold_present` or `structural_damage_severity`) may be inferable from narrative observations. Others (like `elevation_certificate` or `building_description_code`) exist in the underlying dataset but were never mentioned in the adjuster notes -- the LLM should return null for these. This is realistic: you always define your ideal schema, then see what the source text can actually provide.

In [ ]:
# Define the extraction target schema.
# Each field has a type and description -- the descriptions help the LLM 
# understand what to look for in the unstructured text.
# Fields are organized into groups that we'll use for parallel extraction later.

class ClaimExtraction(BaseModel):
    """Structured data extracted from insurance adjuster field notes."""
    
    # --- Group 1: Claim & Location ---
    claim_id: Optional[str] = Field(None, description="The claim identifier or number")
    date_of_loss: Optional[str] = Field(None, description="Date when the flood damage occurred (YYYY-MM-DD format)")
    state: Optional[str] = Field(None, description="Two-letter US state code (e.g., TX, FL, LA)")
    county_code: Optional[str] = Field(None, description="County FIPS code")
    flood_zone: Optional[str] = Field(None, description="FEMA flood zone designation (e.g., AE, V, X, A)")
    
    # --- Group 2: Flood Event & Damage ---
    flood_event: Optional[str] = Field(None, description="Name of the flood event or disaster")
    water_depth_inches: Optional[int] = Field(None, description="Depth of flood water in inches")
    flood_duration_hours: Optional[int] = Field(None, description="How long flood water remained, in hours")
    building_damage_amount: Optional[float] = Field(None, description="Dollar amount of building/structural damage")
    contents_damage_amount: Optional[float] = Field(None, description="Dollar amount of contents/personal property damage")
    building_property_value: Optional[float] = Field(None, description="Assessed or estimated value of the building")
    contents_property_value: Optional[float] = Field(None, description="Assessed or estimated value of contents")
    structural_damage_severity: Optional[str] = Field(None, description="Overall severity of structural damage: minor, moderate, severe, or catastrophic")
    amount_paid_building: Optional[float] = Field(None, description="Insurance payout amount for building damage")
    amount_paid_contents: Optional[float] = Field(None, description="Insurance payout amount for contents damage")
    
    # --- Group 3: Property Details ---
    basement_type: Optional[int] = Field(None, description="Basement/enclosure/crawlspace type code (0-4)")
    floodproofed: Optional[bool] = Field(None, description="Whether the building has floodproofing measures")
    mold_present: Optional[bool] = Field(None, description="Whether mold was observed during inspection")
    number_of_stories: Optional[int] = Field(None, description="Number of stories in the building")
    elevation_certificate: Optional[bool] = Field(None, description="Whether an elevation certificate exists for the property")
    building_description_code: Optional[str] = Field(None, description="Building description type code (e.g., 1=single family, 2=2-4 family)")


# Organize fields into groups for focused extraction prompts.
# Each group gets its own Cortex call with a tailored prompt.
FIELD_GROUPS = {
    "claim_location": {
        "label": "Claim & Location",
        "fields": ["claim_id", "date_of_loss", "state", "county_code", "flood_zone"],
    },
    "flood_damage": {
        "label": "Flood Event & Damage",
        "fields": [
            "flood_event", "water_depth_inches", "flood_duration_hours",
            "building_damage_amount", "contents_damage_amount",
            "building_property_value", "contents_property_value",
            "structural_damage_severity", "amount_paid_building", "amount_paid_contents",
        ],
    },
    "property_details": {
        "label": "Property Details",
        "fields": [
            "basement_type", "floodproofed", "mold_present",
            "number_of_stories", "elevation_certificate", "building_description_code",
        ],
    },
}

ALL_FIELDS = [f for g in FIELD_GROUPS.values() for f in g["fields"]]
print(f"Extraction schema: {len(ALL_FIELDS)} fields in {len(FIELD_GROUPS)} groups")
for name, group in FIELD_GROUPS.items():
    print(f"  {group['label']}: {len(group['fields'])} fields")


# Generate the JSON schema and simplify for Cortex compatibility.
# Pydantic v2 emits anyOf for Optional fields, but Cortex needs a type array like ["string", "null"].
def simplify_schema_for_cortex(schema: dict) -> dict:
    """Convert Pydantic JSON schema to Cortex-compatible format with nullable types."""
    simplified = {"type": "object", "properties": {}}
    for field_name, field_def in schema.get("properties", {}).items():
        prop = {}
        if "anyOf" in field_def:
            types = [opt["type"] for opt in field_def["anyOf"] if "type" in opt]
            prop["type"] = types if len(types) > 1 else types[0]
        elif "type" in field_def:
            prop["type"] = field_def["type"]
        if "description" in field_def:
            prop["description"] = field_def["description"]
        simplified["properties"][field_name] = prop
    return simplified


def schema_for_group(group_key: str) -> dict:
    """Build a Cortex-compatible JSON schema for a single field group."""
    full_schema = ClaimExtraction.model_json_schema()
    group_fields = FIELD_GROUPS[group_key]["fields"]
    full_simplified = simplify_schema_for_cortex(full_schema)
    return {
        "type": "object",
        "properties": {
            k: v for k, v in full_simplified["properties"].items()
            if k in group_fields
        }
    }


extraction_schema = simplify_schema_for_cortex(ClaimExtraction.model_json_schema())
print(f"\nFull schema has {len(extraction_schema['properties'])} properties")
print("Group schemas:")
for gk in FIELD_GROUPS:
    gs = schema_for_group(gk)
    print(f"  {gk}: {len(gs['properties'])} properties")

In [ ]:
# Test the schema with a single extraction call using Cortex structured output.
# The response_format parameter forces the LLM to return valid JSON matching our schema.

sample_note = session.sql("SELECT adjuster_notes FROM CLAIMS_SAMPLE LIMIT 1").collect()[0][0]
print("Input note (first 300 chars):")
print(sample_note[:300] + "...\n")

extraction_prompt = json.dumps([
    {
        "role": "system", 
        "content": "Extract structured insurance claim data from the adjuster notes. \
                    Return all fields you can find. Use null for fields not mentioned in the notes."
    },
    {"role": "user", "content": sample_note}
])

schema_json = json.dumps(extraction_schema)

# Inspect the raw Cortex response structure
raw_result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        PARSE_JSON($${extraction_prompt}$$),
        {{
            'temperature': 0,
            'max_tokens': 2048,
            'response_format': {{
                'type': 'json',
                'schema': PARSE_JSON($${schema_json}$$)
            }}
        }}
    ) AS raw_response
""").collect()[0][0]

print("Raw Cortex response:")
print(raw_result)

### EXERCISE: Add Validation Rules

The basic schema above accepts any value that matches the type. But we know things about our domain -- states are 2-letter codes, water depth can't be negative, damage severity has a fixed set of values. Add Pydantic **field validators** to catch common extraction errors.

Your validators should check:
1. `state` must be exactly 2 uppercase letters
2. `water_depth_inches` must be >= 0 and <= 600 (50 feet max)
3. `building_damage_amount` and `contents_damage_amount` must be >= 0
4. `basement_type` must be between 0 and 4
5. `structural_damage_severity` must be one of: minor, moderate, severe, catastrophic (or null)
6. `number_of_stories` must be between 1 and 5 (or null)

In [ ]:
# EXERCISE: Add field validators to catch extraction errors.

class ClaimExtractionValidated(BaseModel):
    """Extraction schema with domain-specific validation rules."""
    
    # --- Group 1: Claim & Location ---
    claim_id: Optional[str] = Field(None, description="The claim identifier or number")
    date_of_loss: Optional[str] = Field(None, description="Date when the flood damage occurred (YYYY-MM-DD format)")
    state: Optional[str] = Field(None, description="Two-letter US state code")
    county_code: Optional[str] = Field(None, description="County FIPS code")
    flood_zone: Optional[str] = Field(None, description="FEMA flood zone designation (e.g., AE, V, X, A)")
    
    # --- Group 2: Flood Event & Damage ---
    flood_event: Optional[str] = Field(None, description="Name of the flood event or disaster")
    water_depth_inches: Optional[int] = Field(None, description="Depth of flood water in inches")
    flood_duration_hours: Optional[int] = Field(None, description="How long flood water remained, in hours")
    building_damage_amount: Optional[float] = Field(None, description="Dollar amount of building/structural damage")
    contents_damage_amount: Optional[float] = Field(None, description="Dollar amount of contents/personal property damage")
    building_property_value: Optional[float] = Field(None, description="Assessed or estimated value of the building")
    contents_property_value: Optional[float] = Field(None, description="Assessed or estimated value of contents")
    structural_damage_severity: Optional[str] = Field(None, description="Overall severity: minor, moderate, severe, or catastrophic")
    amount_paid_building: Optional[float] = Field(None, description="Insurance payout amount for building damage")
    amount_paid_contents: Optional[float] = Field(None, description="Insurance payout amount for contents damage")
    
    # --- Group 3: Property Details ---
    basement_type: Optional[int] = Field(None, description="Basement/enclosure/crawlspace type code (0-4)")
    floodproofed: Optional[bool] = Field(None, description="Whether the building has floodproofing measures")
    mold_present: Optional[bool] = Field(None, description="Whether mold was observed during inspection")
    number_of_stories: Optional[int] = Field(None, description="Number of stories in the building")
    elevation_certificate: Optional[bool] = Field(None, description="Whether an elevation certificate exists for the property")
    building_description_code: Optional[str] = Field(None, description="Building description type code")

    @field_validator('state')
    @classmethod
    def validate_state(cls, v):
      if v is not None and (len(v) != 2 or not v.isalpha()):
          raise ValueError(f"State must be 2 letters, got: {v}")
      return v.upper() if v else v

    @field_validator('water_depth_inches')
    @classmethod
    def validate_water_depth_inches(cls, v):
      if v is not None and (v < 0):
          raise ValueError(f"Water depth inches can't be negative, got: {v}")
      return v if v else v

    @field_validator('building_damage_amount')
    @classmethod
    def validate_building_damage_amount(cls, v):
      if v is not None and (v < 0):
          raise ValueError(f"Building damage amount can't be negative, got: {v}")
      return v if v else v

    @field_validator('contents_damage_amount')
    @classmethod
    def validate_contents_damage_amount(cls, v):
      if v is not None and (v < 0):
          raise ValueError(f"Contents damage amount can't be negative, got: {v}")
      return v if v else v

    @field_validator('structural_damage_severity')
    @classmethod
    def validate_severity(cls, v):
      if v is not None:
          allowed = {'minor', 'moderate', 'severe', 'catastrophic'}
          if v.lower() not in allowed:
              raise ValueError(f"Severity must be one of {allowed}, got: {v}")
          return v.lower()
      return v

    @field_validator('number_of_stories')
    @classmethod
    def validate_stories(cls, v):
      if v is not None and (v < 1 or v > 5):
          raise ValueError(f"Stories must be 1-5, got: {v}")
      return v

    # TODO: add in a validator for basement_type


# Test your validators -- this should pass:
valid_data = {"state": "TX", "water_depth_inches": 24, "building_damage_amount": 50000, "structural_damage_severity": "moderate"}
print("Valid data test:", ClaimExtractionValidated(**valid_data))

# This should raise a validation error once your basement_type validator is written (try uncommenting):
# invalid_data = {"state": "TX", "water_depth_inches": 5, "basement_type": 5}
# ClaimExtractionValidated(**invalid_data)

These Pydantic models serve double duty in our LangGraph pipeline:
- The **JSON schema** feeds into Cortex's `response_format` to constrain the LLM's output structure
- The **validators** run in the VALIDATE node to catch extraction errors before they reach your database

Notice we organized fields into three groups: **Claim & Location**, **Flood Event & Damage**, and **Property Details**. In the next section, we'll use these groups to build a pipeline where each group gets its own focused extraction prompt via LangGraph's **Send API**.

---
# Section 3: Building the Extraction Pipeline

Now we put it all together. Instead of cramming all 21 fields into a single prompt, we'll split extraction into **focused groups** using LangGraph's **Send API**. Each group gets its own Cortex call with a prompt tailored to just those fields.

**Why group?** As schemas grow beyond ~15-20 fields, single-prompt extraction degrades -- the LLM juggles too many field definitions, confuses similar fields (e.g., building damage vs contents damage vs building payout), and you can't tune prompts per domain area. Grouping keeps each prompt focused and lets you add groups independently.

The pipeline architecture:

```
              +─────────+
              |  START   |
              +────┬─────+
                   |
          +────────┼────────+
          |        |        |   Fan-out via Send()
    +─────v──+ +──v─────+ +v────────+
    | Group1 | | Group2 | | Group3  |   3 parallel extract nodes
    |Claim & | |Flood & | |Property |
    |Location| |Damage  | |Details  |
    +────┬───+ +───┬────+ +───┬─────+
         |         |          |
         +─────────┼──────────+
              +────v─────+
              |  MERGE   |   Combine all group results
              +────┬─────+
              +────v─────+
              | VALIDATE |   Full-schema Pydantic validation
              +────┬─────+
              +────┴─────+
           (valid)    (invalid)
              |          |
         +────v───+  +──v──+
         |FINALIZE|  | FIX | -> re-validate -> ...
         +────────+  +─────+
```

> **Note on parallelism**: In Snowpark notebooks, SQL calls are synchronous, so the three Send targets run sequentially. The Send pattern still matters because: (1) each group gets a focused prompt, (2) groups can have specialized instructions, and (3) in environments with async I/O the same graph runs in parallel without code changes.

In [ ]:
# Step 1: Define the graph state.
# We need two state types:
#   - ExtractionState: the main pipeline state with reducers for parallel fan-in
#   - GroupWorkerState: minimal state passed to each group extraction worker via Send

class ExtractionState(TypedDict):
    # --- Inputs (set once at invocation) ---
    claim_id: str                    # Unique identifier for tracking
    adjuster_notes: str              # Raw unstructured text to extract from
    
    # --- Group extraction results (accumulated via reducer) ---
    group_results: Annotated[list, operator.add]  # Each worker appends its group result
    
    # --- Merged extraction (after all groups complete) ---
    raw_extraction: dict             # Combined extraction from all groups
    validated_extraction: dict       # After Pydantic validation passes
    extraction_errors: list          # Validation errors (if any)
    
    # --- Control flow ---
    retry_count: int                 # Number of fix attempts so far
    max_retries: int                 # Maximum retries before giving up
    is_valid: bool                   # Whether current extraction passes validation
    
    # --- Observability ---
    node_history: Annotated[list, operator.add]  # Workers append in parallel
    cortex_calls: int                # Total LLM calls made
    processing_time: float           # Total wall-clock seconds


class GroupWorkerState(TypedDict):
    """Minimal state passed to each group extraction worker via Send."""
    claim_id: str
    adjuster_notes: str
    group_key: str         # Which group this worker handles
    group_fields: list     # Field names in this group
    group_label: str       # Human-readable group name

Notice the `Annotated[list, operator.add]` on `group_results` and `node_history`. This is LangGraph's **reducer** pattern -- when multiple Send workers return results simultaneously, the reducer tells LangGraph how to combine them. `operator.add` means "concatenate the lists," so each worker's results get merged automatically.

The observability fields (`node_history`, `cortex_calls`, `processing_time`) are critical for:
- **Cost tracking**: knowing how many LLM calls each claim required, and which groups triggered them
- **Debugging**: seeing the exact path through the graph, including which groups ran
- **Performance analysis**: measuring wall-clock time per claim and per group

In [ ]:
# Step 2: Implement the EXTRACT nodes.
# Instead of a single extract node, we fan out to one worker per field group.

def fan_out_to_groups(state: ExtractionState) -> list:
    """Route to one extraction worker per field group using LangGraph Send."""
    return [
        Send("extract_group", {
            "claim_id": state["claim_id"],
            "adjuster_notes": state["adjuster_notes"],
            "group_key": group_key,
            "group_fields": group_info["fields"],
            "group_label": group_info["label"],
        })
        for group_key, group_info in FIELD_GROUPS.items()
    ]


def extract_group_node(state: GroupWorkerState) -> dict:
    """Extract fields for a single group. Called once per group via Send."""
    start_time = time.time()
    group_key = state["group_key"]
    fields = state["group_fields"]
    label = state["group_label"]
    
    # Build a focused prompt for just this group's fields
    field_descriptions = "\n".join(
        f"- {f}: {ClaimExtractionValidated.model_fields[f].description}"
        for f in fields
    )
    
    extraction_prompt = json.dumps([
        {
            "role": "system",
            "content": (
                f"You are an expert insurance data extraction system. "
                f"Extract ONLY the following {label} fields from the adjuster notes. "
                f"Be precise with numbers, dates, and codes. "
                f"If a field is not mentioned or cannot be reasonably inferred from the notes, use null.\n\n"
                f"Fields to extract:\n{field_descriptions}"
            )
        },
        {"role": "user", "content": state["adjuster_notes"]}
    ])
    
    group_schema = json.dumps(schema_for_group(group_key))
    
    # Tag with group info for per-group cost tracking
    session.sql(
        f"ALTER SESSION SET QUERY_TAG = "
        f"'{{\"lab\": \"claims-extraction\", \"section\": \"extract\", "
        f"\"node\": \"extract_group\", \"group\": \"{group_key}\"}}'"
    ).collect()
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON($${extraction_prompt}$$),
            {{
                'temperature': 0,
                'response_format': {{
                    'type': 'json',
                    'schema': PARSE_JSON($${group_schema}$$)
                }}
            }}
        ):structured_output[0]:raw_message::STRING AS result
    """).collect()[0][0]
    
    extracted = json.loads(result)
    elapsed = time.time() - start_time
    
    non_null = sum(1 for v in extracted.values() if v is not None)
    print(f"    [{label}] {non_null}/{len(fields)} fields extracted ({elapsed:.1f}s)")
    
    # Note: cortex_calls and processing_time are embedded in group_results
    # because Send workers can't safely write to non-reducer state fields.
    # The merge node will tally these up.
    return {
        "group_results": [{
            "group_key": group_key,
            "fields": extracted,
            "cortex_calls": 1,
            "processing_time": elapsed,
        }],
        "node_history": [f"extract_{group_key}"],
    }

In [ ]:
# Step 3: Implement the VALIDATE node.
# This node runs Pydantic validation -- no LLM calls, just business rules.

def validate_node(state: ExtractionState) -> dict:
    """Validate extracted data using the Pydantic model with domain rules."""
    errors = []
    raw = state["raw_extraction"]
    
    try:
        validated = ClaimExtractionValidated(**raw)
        return {
            "validated_extraction": validated.model_dump(),
            "extraction_errors": [],
            "is_valid": True,
            "node_history": ["validate_pass"]
        }
    except Exception as e:
        # Parse Pydantic validation errors into a structured list
        # that we can feed back to the LLM in the FIX node
        if hasattr(e, 'errors'):
            for err in e.errors():
                errors.append({
                    "field": ".".join(str(x) for x in err["loc"]),
                    "message": err["msg"],
                    "type": err["type"],
                    "input_value": str(err.get("input", ""))[:100]
                })
        else:
            errors.append({"field": "unknown", "message": str(e), "type": "general"})
        
        return {
            "extraction_errors": errors,
            "is_valid": False,
            "node_history": ["validate_fail"]
        }

In [ ]:
# Step 4: Implement the FIX node.
# When validation fails, we re-prompt the LLM with the specific errors.
# This is much more effective than just retrying blindly.

def fix_node(state: ExtractionState) -> dict:
    """Re-prompt Cortex with specific validation errors to fix the extraction."""
    start_time = time.time()
    
    # Format the errors into a clear description for the LLM
    error_descriptions = "\n".join([
        f"- Field '{e['field']}': {e['message']} (current value: {e['input_value']})"
        for e in state["extraction_errors"]
    ])
    
    fix_prompt = json.dumps([
        {
            "role": "system", 
            "content": (
                "You are an expert insurance data extraction system. "
                "A previous extraction attempt had validation errors. "
                "Fix ONLY the fields with errors. "
                "Keep all correctly extracted fields unchanged."
            )
        },
        {
            "role": "user", 
            "content": (
                f"Original adjuster notes:\n{state['adjuster_notes']}\n\n"
                f"Previous extraction (with errors):\n{json.dumps(state['raw_extraction'], indent=2)}\n\n"
                f"Validation errors to fix:\n{error_descriptions}\n\n"
                "Please provide the corrected extraction."
            )
        }
    ])
    
    schema_json = json.dumps(simplify_schema_for_cortex(ClaimExtractionValidated.model_json_schema()))
    
    session.sql(
        "ALTER SESSION SET QUERY_TAG = "
        "'{\"lab\": \"claims-extraction\", \"section\": \"extract\", \"node\": \"fix\"}'"
    ).collect()
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON($${fix_prompt}$$),
            {{
                'temperature': 0,
                'max_tokens': 2048,
                'response_format': {{
                    'type': 'json',
                    'schema': PARSE_JSON($${schema_json}$$)
                }}
            }}
        ):structured_output[0]:raw_message::STRING AS result
    """).collect()[0][0]
    
    extracted = json.loads(result)
    elapsed = time.time() - start_time
    
    return {
        "raw_extraction": extracted,
        "retry_count": state.get("retry_count", 0) + 1,
        "cortex_calls": state.get("cortex_calls", 0) + 1,
        "node_history": ["fix"],
        "processing_time": state.get("processing_time", 0) + elapsed
    }

In [ ]:
# Step 5: Implement the FINALIZE node.
# Packages the result with metadata for downstream analysis.

def finalize_node(state: ExtractionState) -> dict:
    """Finalize the extraction result with metadata."""
    result = state.get("validated_extraction") or state.get("raw_extraction", {})
    
    # Attach processing metadata to the result
    result["_metadata"] = {
        "claim_id": state["claim_id"],
        "is_valid": state.get("is_valid", False),
        "retry_count": state.get("retry_count", 0),
        "cortex_calls": state.get("cortex_calls", 0),
        "node_path": " -> ".join(state.get("node_history", []) + ["finalize"]),
        "processing_time_seconds": round(state.get("processing_time", 0), 2),
        "had_errors": len(state.get("extraction_errors", [])) > 0
    }
    
    return {
        "validated_extraction": result,
        "node_history": ["finalize"]
    }

In [ ]:
# Step 6: Define the merge node, routing function, and assemble the graph.

def merge_groups_node(state: ExtractionState) -> dict:
    """Merge results from all group extraction workers into one dict."""
    merged = {}
    total_calls = 0
    total_time = 0.0
    
    for group_result in state.get("group_results", []):
        merged.update(group_result["fields"])
        total_calls += group_result.get("cortex_calls", 0)
        total_time += group_result.get("processing_time", 0)
    
    return {
        "raw_extraction": merged,
        "cortex_calls": total_calls,
        "processing_time": total_time,
        "node_history": ["merge"],
    }


def should_retry(state: ExtractionState) -> Literal["fix", "finalize"]:
    """Decide whether to retry extraction or finalize with what we have."""
    if state.get("is_valid", False):
        return "finalize"
    elif state.get("retry_count", 0) < state.get("max_retries", 2):
        return "fix"
    else:
        # Max retries exhausted -- finalize with whatever we have
        return "finalize"


# Build the graph
workflow = StateGraph(ExtractionState)

# Add nodes
workflow.add_node("extract_group", extract_group_node)
workflow.add_node("merge", merge_groups_node)
workflow.add_node("validate", validate_node)
workflow.add_node("fix", fix_node)
workflow.add_node("finalize", finalize_node)

# Fan-out: START dispatches one Send per field group
workflow.add_conditional_edges(
    START,
    fan_out_to_groups,
    ["extract_group"]
)

# All group workers converge into merge
workflow.add_edge("extract_group", "merge")

# After merge: validate, then conditional retry
workflow.add_edge("merge", "validate")
workflow.add_conditional_edges(
    "validate",
    should_retry,
    {"fix": "fix", "finalize": "finalize"}
)
workflow.add_edge("fix", "validate")    # After fix, re-validate (creates the retry loop)
workflow.add_edge("finalize", END)

# Compile the graph
extraction_graph = workflow.compile()

print("Grouped extraction pipeline compiled!")
print(f"Groups: {[g['label'] for g in FIELD_GROUPS.values()]}")
print(f"Flow: START -> fan_out(3 groups) -> merge -> validate -> fix/finalize")

In [ ]:
# Step 7: Test the graph with a single claim!
test_claim = session.sql(
    "SELECT claim_id, adjuster_notes FROM CLAIMS_SAMPLE LIMIT 1"
).collect()[0]

initial_state = {
    "claim_id": str(test_claim[0]),
    "adjuster_notes": test_claim[1],
    "group_results": [],
    "raw_extraction": {},
    "validated_extraction": {},
    "extraction_errors": [],
    "retry_count": 0,
    "max_retries": 2,
    "is_valid": False,
    "node_history": [],
    "cortex_calls": 0,
    "processing_time": 0.0
}

print(f"Processing claim {initial_state['claim_id']}...")
print(f"Note preview: {initial_state['adjuster_notes'][:200]}...\n")
print(f"Extracting {len(ALL_FIELDS)} fields across {len(FIELD_GROUPS)} groups...\n")

result = extraction_graph.invoke(initial_state)

print(f"\nNode path: {' -> '.join(result['node_history'])}")
print(f"Valid: {result['is_valid']}")
print(f"Cortex calls: {result['cortex_calls']} ({len(FIELD_GROUPS)} extract + {result['retry_count']} fix)")
print(f"Processing time: {result['processing_time']:.2f}s")

# Show extracted data with null/non-null summary
extracted = result['validated_extraction']
non_null = sum(1 for k, v in extracted.items() if v is not None and k != '_metadata')
print(f"\nExtracted {non_null}/{len(ALL_FIELDS)} fields (non-null):")
print(json.dumps({k: v for k, v in extracted.items() if k != '_metadata'}, indent=2))

### CHALLENGE: Add a Confidence Scoring Node

The current pipeline extracts and validates, but doesn't tell us *how confident* we should be in the results. Add a `confidence_score` node between VALIDATE and FINALIZE that:

1. Counts how many fields were extracted (non-null count out of the total fields)
2. Assigns a confidence level:
   - **high**: > 80% of fields extracted AND validation passed
   - **medium**: 50-80% of fields extracted
   - **low**: < 50% of fields extracted
3. Adds a `confidence` key to the extraction result

Think about how fields like `elevation_certificate` and `building_description_code` affect this metric -- they'll almost always be null because they're not in the notes. Should you exclude "expected null" fields from the confidence calculation?

In [ ]:
# CHALLENGE: Implement your confidence scoring node here.

# def confidence_score_node(state: ExtractionState) -> dict:
#     """Score confidence based on extraction completeness."""
#     extraction = state.get("validated_extraction") or state.get("raw_extraction", {})
#     
#     # Count non-null fields (excluding _metadata)
#     total_fields = len(ALL_FIELDS)
#     extracted_fields = sum(1 for k, v in extraction.items() 
#                          if v is not None and k != "_metadata" and k in ALL_FIELDS)
#     completeness = extracted_fields / total_fields
#     
#     if completeness > 0.8 and state.get("is_valid", False):
#         confidence = "high"
#     elif completeness > 0.5:
#         confidence = "medium"
#     else:
#         confidence = "low"
#     
#     extraction["_confidence"] = {
#         "level": confidence,
#         "completeness": round(completeness, 2),
#         "extracted_fields": extracted_fields,
#         "total_fields": total_fields
#     }
#     
#     return {
#         "validated_extraction": extraction,
#         "node_history": ["confidence_score"]
#     }

### Section 3 Recap

We built a grouped extraction pipeline with LangGraph's Send API:

- **Fan-out**: `fan_out_to_groups` dispatches one `Send()` per field group, each with a focused prompt
- **Merge**: `merge_groups_node` combines results from all groups into a single extraction dict
- **Validate + Fix**: Pydantic validation catches errors; the FIX node re-prompts with specific error details
- **Observability**: every node logs its activity via `node_history`, and per-group Cortex calls are tracked via query tags

The grouped approach trades more Cortex calls for more focused prompts. Each group's prompt only asks about 5-10 fields instead of all 21, reducing confusion between similar fields (e.g., `building_damage_amount` vs `amount_paid_building`).

---
# Section 4: Running Extraction at Scale

Processing a single claim is straightforward. Processing hundreds or thousands requires careful engineering:

- **Progress tracking** -- know what's been processed so you can resume after interruptions
- **Error isolation** -- one bad claim shouldn't kill the entire batch
- **Result persistence** -- save results to Snowflake tables, not just Python variables
- **Idempotency** -- re-running a cell shouldn't create duplicates

> **Snowflake Feature**: We'll use `MERGE` statements for idempotent progress tracking and `VARIANT` columns to store the extracted JSON alongside metadata.

In [ ]:
-- Create tables for results and progress tracking.
-- Using IF NOT EXISTS makes this safe to re-run after a kernel restart.

CREATE TABLE IF NOT EXISTS EXTRACTION_RESULTS (
    claim_id STRING,
    extracted_data VARIANT,
    is_valid BOOLEAN,
    retry_count INTEGER,
    cortex_calls INTEGER,
    processing_time_seconds FLOAT,
    node_path STRING,
    extraction_errors VARIANT,
    extracted_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS EXTRACTION_PROGRESS (
    claim_id STRING PRIMARY KEY,
    status STRING,          -- 'completed', 'failed', 'in_progress'
    started_at TIMESTAMP_NTZ,
    completed_at TIMESTAMP_NTZ
);

SELECT 'Tables ready' AS status,
    (SELECT COUNT(*) FROM EXTRACTION_RESULTS) AS existing_results,
    (SELECT COUNT(*) FROM EXTRACTION_PROGRESS) AS existing_progress;

In [ ]:
# Batch extraction function with checkpointing and error isolation.
# Key design decisions:
#   1. Query for UNPROCESSED claims (skips already-completed ones on restart)
#   2. MERGE for idempotent progress tracking (no duplicates on re-run)
#   3. try/except per claim (one failure doesn't stop the batch)
#   4. Results saved to Snowflake table after EACH claim (not just at the end)

def run_batch_extraction(batch_size=10, max_retries=2):
    """Run extraction on unprocessed claims with per-claim checkpointing."""
    
    # Find claims that haven't been processed yet -- this is the restartability key
    unprocessed = session.sql(f"""
        SELECT an.claim_id, an.adjuster_notes
        FROM ADJUSTER_NOTES an
        LEFT JOIN EXTRACTION_PROGRESS ep ON an.claim_id::STRING = ep.claim_id
        WHERE ep.claim_id IS NULL OR ep.status = 'failed'
        LIMIT {batch_size}
    """).collect()
    
    total = len(unprocessed)
    if total == 0:
        print("All claims already processed! Nothing to do.")
        return [], []
    
    print(f"Processing {total} unprocessed claims (batch_size={batch_size})...\n")
    results = []
    errors = []
    
    for i, row in enumerate(unprocessed):
        claim_id = str(row[0])
        notes = row[1]
        print(f"  [{i+1}/{total}] Claim {claim_id}...", flush=True)
        
        # Mark as in-progress using MERGE (idempotent)
        session.sql(f"""
            MERGE INTO EXTRACTION_PROGRESS ep
            USING (SELECT '{claim_id}' AS claim_id) src
            ON ep.claim_id = src.claim_id
            WHEN MATCHED THEN UPDATE SET status='in_progress', started_at=CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (claim_id, status, started_at)
                VALUES (src.claim_id, 'in_progress', CURRENT_TIMESTAMP())
        """).collect()
        
        try:
            state = {
                "claim_id": claim_id,
                "adjuster_notes": notes,
                "group_results": [],
                "raw_extraction": {},
                "validated_extraction": {},
                "extraction_errors": [],
                "retry_count": 0,
                "max_retries": max_retries,
                "is_valid": False,
                "node_history": [],
                "cortex_calls": 0,
                "processing_time": 0.0
            }
            
            result = extraction_graph.invoke(state)
            
            # Save result to table
            extracted_json = json.dumps(result['validated_extraction']).replace("'", "''")
            errors_json = json.dumps(result.get('extraction_errors', [])).replace("'", "''")
            node_path = ' -> '.join(result['node_history'])
            
            session.sql(f"""
                INSERT INTO EXTRACTION_RESULTS
                (claim_id, extracted_data, is_valid, retry_count, cortex_calls, 
                 processing_time_seconds, node_path, extraction_errors)
                SELECT
                    '{claim_id}',
                    PARSE_JSON('{extracted_json}'),
                    {result['is_valid']},
                    {result['retry_count']},
                    {result['cortex_calls']},
                    {result['processing_time']:.2f},
                    '{node_path}',
                    PARSE_JSON('{errors_json}')
            """).collect()
            
            # Mark completed
            session.sql(f"""
                UPDATE EXTRACTION_PROGRESS
                SET status='completed', completed_at=CURRENT_TIMESTAMP()
                WHERE claim_id='{claim_id}'
            """).collect()
            
            status = "VALID" if result['is_valid'] else "INVALID"
            print(f"    {status} ({result['cortex_calls']} calls, {result['processing_time']:.1f}s)")
            results.append(result)
            
        except Exception as e:
            print(f"    FAILED: {str(e)[:80]}")
            session.sql(f"""
                UPDATE EXTRACTION_PROGRESS
                SET status='failed', completed_at=CURRENT_TIMESTAMP()
                WHERE claim_id='{claim_id}'
            """).collect()
            errors.append({"claim_id": claim_id, "error": str(e)})
    
    print(f"\nBatch complete: {len(results)} succeeded, {len(errors)} failed")
    return results, errors

In [ ]:
# Run a small initial batch to verify everything works end-to-end
results, errors = run_batch_extraction(batch_size=10, max_retries=2)

### EXPERIMENT: Batch Size

Try running with `batch_size=20` in the cell below. Does it take roughly twice as long as the batch of 10? Why or why not?

Consider: each claim is processed **sequentially** in our Python loop. What would happen if we could parallelize? (We'll explore this in the Performance section.)

In [ ]:
# Process the remaining claims. Adjust batch_size based on your time budget.
# Because of restartability, this picks up where the previous batch left off.
results, errors = run_batch_extraction(batch_size=100, max_retries=2)

In [ ]:
-- Check overall extraction progress
SELECT
    status,
    COUNT(*) AS claim_count,
    ROUND(AVG(DATEDIFF('second', started_at, completed_at)), 1) AS avg_duration_seconds
FROM EXTRACTION_PROGRESS
GROUP BY status
ORDER BY claim_count DESC;

In [ ]:
-- Preview extraction results using Snowflake's VARIANT : notation
-- This is a powerful feature for querying semi-structured data stored as JSON
SELECT
    claim_id,
    is_valid,
    retry_count,
    cortex_calls,
    ROUND(processing_time_seconds, 1) AS time_sec,
    node_path,
    extracted_data:state::STRING AS extracted_state,
    extracted_data:building_damage_amount::FLOAT AS extracted_bldg_damage,
    extracted_data:water_depth_inches::INTEGER AS extracted_water_depth,
    extracted_data:flood_zone::STRING AS extracted_flood_zone
FROM EXTRACTION_RESULTS
ORDER BY extracted_at DESC
LIMIT 10;

---
# Section 5: Quick Accuracy Check -- Eval Lab Preview

Because we generated our adjuster notes from structured data, we have **ground truth** -- we know exactly what the correct extracted values should be. Let's do a quick accuracy check by comparing extracted values to the originals.

**Important nuance**: Some of our new fields (`mold_present`, `structural_damage_severity`, `number_of_stories`) were NOT in the original structured data -- they were invented by the LLM when generating the notes. We can't evaluate accuracy for those. Other fields (`elevation_certificate`, `building_description_code`, `amount_paid_building`, `amount_paid_contents`) exist in the underlying data but were NOT mentioned in the notes -- the correct extraction for these is null.

> This is a preview. The next lab will build a comprehensive evaluation framework with precision/recall, confusion matrices, and regression metrics.

In [ ]:
-- Join extracted results to ground truth and compute field-level accuracy.
-- For text fields: exact match. For numbers: within 1% tolerance.
-- Fields not in the adjuster notes (elevation_certificate, etc.) are tracked separately.
CREATE OR REPLACE VIEW EXTRACTION_ACCURACY AS
SELECT
    er.claim_id,
    er.is_valid,
    er.retry_count,
    er.cortex_calls,
    
    -- Exact match checks (text/categorical fields present in the notes)
    CASE WHEN er.extracted_data:state::STRING = cs.state THEN 1 ELSE 0 END AS state_match,
    CASE WHEN er.extracted_data:flood_zone::STRING = cs.flood_zone THEN 1 ELSE 0 END AS flood_zone_match,
    CASE WHEN er.extracted_data:flood_event::STRING = cs.flood_event THEN 1 ELSE 0 END AS flood_event_match,
    CASE WHEN er.extracted_data:water_depth_inches::INTEGER = cs.water_depth THEN 1 ELSE 0 END AS water_depth_match,
    CASE WHEN er.extracted_data:basement_type::INTEGER = cs.basement_type THEN 1 ELSE 0 END AS basement_type_match,
    
    -- Numeric proximity checks (within 1% tolerance for dollar amounts)
    CASE WHEN ABS(er.extracted_data:building_damage_amount::FLOAT - cs.building_damage_amount)
         / NULLIF(cs.building_damage_amount, 0) < 0.01 THEN 1 ELSE 0 END AS building_damage_match,
    CASE WHEN ABS(er.extracted_data:contents_damage_amount::FLOAT - cs.contents_damage_amount)
         / NULLIF(cs.contents_damage_amount, 0) < 0.01 THEN 1 ELSE 0 END AS contents_damage_match,
    CASE WHEN ABS(er.extracted_data:building_property_value::FLOAT - cs.building_property_value)
         / NULLIF(cs.building_property_value, 0) < 0.01 THEN 1 ELSE 0 END AS property_value_match,
    
    -- Fields NOT in the notes (should be null -- check for hallucination)
    CASE WHEN er.extracted_data:elevation_certificate IS NULL THEN 1 ELSE 0 END AS elevation_cert_correctly_null,
    CASE WHEN er.extracted_data:building_description_code IS NULL THEN 1 ELSE 0 END AS bldg_desc_correctly_null,
    CASE WHEN er.extracted_data:amount_paid_building IS NULL THEN 1 ELSE 0 END AS paid_bldg_correctly_null,
    CASE WHEN er.extracted_data:amount_paid_contents IS NULL THEN 1 ELSE 0 END AS paid_contents_correctly_null,
    
    -- Narrative-inferred fields (no ground truth -- just show what was extracted)
    er.extracted_data:mold_present::BOOLEAN AS extracted_mold_present,
    er.extracted_data:structural_damage_severity::STRING AS extracted_severity,
    er.extracted_data:number_of_stories::INTEGER AS extracted_stories

FROM EXTRACTION_RESULTS er
JOIN CLAIMS_SAMPLE cs ON er.claim_id = cs.claim_id::STRING;

In [ ]:
-- Aggregate accuracy across all processed claims
SELECT
    COUNT(*) AS total_claims,
    SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) AS valid_extractions,
    
    -- Fields present in the notes (should match ground truth)
    ROUND(AVG(state_match) * 100, 1) AS state_accuracy_pct,
    ROUND(AVG(flood_zone_match) * 100, 1) AS flood_zone_accuracy_pct,
    ROUND(AVG(flood_event_match) * 100, 1) AS flood_event_accuracy_pct,
    ROUND(AVG(water_depth_match) * 100, 1) AS water_depth_accuracy_pct,
    ROUND(AVG(basement_type_match) * 100, 1) AS basement_type_accuracy_pct,
    ROUND(AVG(building_damage_match) * 100, 1) AS building_damage_accuracy_pct,
    ROUND(AVG(contents_damage_match) * 100, 1) AS contents_damage_accuracy_pct,
    ROUND(AVG(property_value_match) * 100, 1) AS property_value_accuracy_pct,
    
    -- Fields NOT in notes (should be null -- higher = fewer hallucinations)
    ROUND(AVG(elevation_cert_correctly_null) * 100, 1) AS elevation_cert_null_pct,
    ROUND(AVG(bldg_desc_correctly_null) * 100, 1) AS bldg_desc_null_pct,
    ROUND(AVG(paid_bldg_correctly_null) * 100, 1) AS paid_bldg_null_pct,
    ROUND(AVG(paid_contents_correctly_null) * 100, 1) AS paid_contents_null_pct,
    
    ROUND(AVG(retry_count), 2) AS avg_retries,
    ROUND(AVG(cortex_calls), 2) AS avg_cortex_calls
FROM EXTRACTION_ACCURACY;

### EXERCISE: Which Fields Are Hardest to Extract?

Write a query that shows per-field accuracy ranked from worst to best. Then answer:
- Why might some fields be harder to extract than others?
- Does the accuracy differ for claims that required retries vs those that didn't?
- What could you change in the extraction prompt to improve the worst-performing fields?

**Hallucination check**: Look at the `*_correctly_null` columns. If `elevation_cert_null_pct` or `bldg_desc_null_pct` is below 100%, the model is fabricating values for fields that aren't in the source text. This is a common problem in production extraction pipelines -- your schema asks for a field, and the model invents a plausible-sounding answer rather than returning null.

In [ ]:
-- EXERCISE: Write your accuracy analysis queries here.
-- 
-- Hint: You can UNPIVOT the accuracy columns to compare fields side by side:
-- SELECT field_name, accuracy FROM (
--     SELECT 'state' AS field_name, AVG(state_match)*100 AS accuracy FROM EXTRACTION_ACCURACY
--     UNION ALL
--     SELECT 'flood_zone', AVG(flood_zone_match)*100 FROM EXTRACTION_ACCURACY
--     ...
-- ) ORDER BY accuracy ASC;
-- 
-- Bonus: Compare accuracy WHERE retry_count = 0 vs retry_count > 0

-- YOUR CODE HERE


### Coming Up: The Eval Lab

This was a quick accuracy check using exact-match and tolerance-based comparisons. In the **next lab (Model Evaluation)**, we'll build a comprehensive evaluation framework covering:

- **Precision and recall** per field
- **Confusion matrices** for categorical fields (state, flood zone)
- **Regression metrics** (MAE, MAPE) for numeric fields
- **Semantic similarity** for text fields (flood event names)
- **Error analysis** — systematic patterns in what the model gets wrong

The tables you created here — `EXTRACTION_RESULTS` and `CLAIMS_SAMPLE` — are the inputs to that lab. Don't drop them!

---
# Section 6: Cost, Performance & Scaling Notes

The sections above covered extraction pipeline design and accuracy. In production, you'd also need to think about **cost**, **performance**, and **scaling**. Here's a quick reference.

### Cost Tracking with Query Tags

Throughout this lab, every Cortex call was tagged with structured JSON metadata via `ALTER SESSION SET QUERY_TAG`. This lets you query `INFORMATION_SCHEMA.QUERY_HISTORY` to attribute costs by section, node, group, etc. -- all without external tooling.

Key takeaways:
- **`temperature=0`** reduces retries, saving credits
- **Structured output** (`response_format`) eliminates JSON parsing failures entirely
- **The retry pattern trades cost for quality** -- track retry rates to find the balance
- **Grouped extraction costs more per claim** (~3 extract calls vs 1) but improves accuracy on complex schemas by keeping prompts focused

### Performance: Python Loop vs SQL Batch vs Hybrid

| Approach | How It Works | Validation | Best For |
|----------|-------------|------------|----------|
| **Python loop** (this lab) | Sequential, one claim at a time | Full Pydantic + retry | Small batches, max quality |
| **SQL batch** (`SELECT ... CORTEX.COMPLETE ... FROM table`) | Snowflake parallelizes across rows | None (raw JSON output) | Max throughput, simple schemas |
| **Hybrid** | SQL batch extract, then Python validate, then LangGraph fix only failures | Full Pydantic, retry only failures | **Production** -- fast AND reliable |

### Scaling to 2.7M Claims

- **SQL CTAS**: `CREATE TABLE AS SELECT` with `CORTEX.COMPLETE` -- Snowflake handles parallelism automatically
- **Warehouse sizing**: Larger warehouses = more parallel Cortex calls = higher throughput (but same total credits)
- **Multi-cluster warehouses** (Enterprise Edition): Auto-scale out with multiple clusters for concurrent queries
- **Partitioned processing**: Split by state or year, process partitions independently for natural checkpointing and cost attribution

In [ ]:
-- Example: Cost breakdown by pipeline section and group.
-- Run this after processing a batch to see where credits went.
SELECT
    PARSE_JSON(query_tag):section::STRING AS lab_section,
    PARSE_JSON(query_tag):node::STRING AS graph_node,
    PARSE_JSON(query_tag):group::STRING AS field_group,
    COUNT(*) AS query_count,
    ROUND(SUM(credits_used_cloud_services), 4) AS total_credits,
    ROUND(AVG(total_elapsed_time) / 1000, 1) AS avg_time_seconds
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    DATE_RANGE_START => DATEADD('hour', -6, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10000
))
WHERE query_tag LIKE '%claims-extraction%'
  AND query_text ILIKE '%CORTEX%COMPLETE%'
GROUP BY lab_section, graph_node, field_group
ORDER BY total_credits DESC;

### What's Next

In the **Model Evaluation lab**, you'll build a comprehensive evaluation framework on top of the `EXTRACTION_RESULTS` and `CLAIMS_SAMPLE` tables created here. That lab covers precision/recall, confusion matrices, regression metrics, semantic similarity, and systematic error analysis.

The tables are ready -- don't drop them!